# Tip handling on the v1 STAR driver

start date: 2026-09-23

Four tip carriers, one tip size each. Two racks per carrier with tips taken out at random, one
holding its right half, one full. Then pick-ups, returns, drops and discards - including tips of
different kinds in one call.

In [ ]:
import datetime
import logging

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO

simulation = True  # True or False

protocol_mode = "simulation" if simulation else "execution"

run_identifier = "v1_star_tips"

started_at = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_path = f"_logs/{protocol_mode}/{run_identifier}_{started_at}.log"
pylabrobot.setup_logger(log_path, level=LOG_LEVEL_IO)
pylabrobot.verbose(True, level=LOG_LEVEL_IO)

print(f"logging to {log_path}")

In [ ]:
from pylabrobot.hamilton import STAR

star = STAR(
  simulation=simulation,
)

await star.setup()

In [ ]:
from pylabrobot.resources import set_tip_tracking

set_tip_tracking(True)

In [ ]:
from pylabrobot.visualizer3D.server import Viewer3D

viewer = Viewer3D(star, name="v1_star_tips")
await viewer.start()

## The racks

In [ ]:
from pylabrobot.resources.hamilton import (
  TIP_CAR_480_A00,
  hamilton_96_tiprack_10uL,
  hamilton_96_tiprack_50uL,
  hamilton_96_tiprack_300uL,
  hamilton_96_tiprack_1000uL,
)

racks = {}
for size, make_rack, track in (
  ("10uL", hamilton_96_tiprack_10uL, 1),
  ("50uL", hamilton_96_tiprack_50uL, 7),
  ("300uL", hamilton_96_tiprack_300uL, 13),
  ("1000uL", hamilton_96_tiprack_1000uL, 19),
):
  carrier = TIP_CAR_480_A00(name=f"carrier_{size}")
  for slot in range(4):
    carrier[slot] = racks[size, slot] = make_rack(name=f"{size}_{slot}")
  star.deck.assign_child_resource(carrier, track=track)

In [ ]:
import random

wells = [f"{row}{column}" for column in range(1, 13) for row in "ABCDEFGH"]
dice = random.Random(0)

for (size, slot), rack in racks.items():
  if slot < 2:  # tips taken out at random
    kept = set(dice.sample(wells, 60))
    rack.set_tip_state({well: well in kept for well in wells})
  elif slot == 2:  # the right half only
    rack.set_tip_state({well: int(well[1:]) > 6 for well in wells})

{
  name: sum(1 for spot in rack.get_all_items() if spot.has_tip())
  for name, rack in ((rack.name, rack) for rack in racks.values())
}

In [ ]:
full_10, full_50, full_300, full_1000 = (
  racks[size, 3] for size in ("10uL", "50uL", "300uL", "1000uL")
)
holed_300, holed_1000 = racks["300uL", 0], racks["1000uL", 1]
right_half_50 = racks["50uL", 2]

## One kind

In [ ]:
await star.pipettes.pick_up_tips([full_300.get_item(f"{row}1") for row in "ABCDEFGH"])

In [ ]:
await star.pipettes.request_tool_bottom_z_positions()

In [ ]:
await star.pipettes.return_tips()

In [ ]:
await star.pipettes.request_stop_disc_z_positions()

In [ ]:
# a rack with tips taken out at random: what is left is not a column
await star.pipettes.pick_up_tips([spot for spot in holed_300.get_all_items() if spot.has_tip()][:8])

In [ ]:
(
  await star.pipettes.request_stop_disc_z_positions(),
  await star.pipettes.request_tool_bottom_z_positions(),
)

In [ ]:
await star.pipettes.return_tips()

In [ ]:
await star.pipettes.pick_up_tips([right_half_50.get_item(f"{row}12") for row in "ABCDEFGH"])

In [ ]:
await star.pipettes.return_tips()

## Two kinds in one call

One command per kind, in ascending X.

In [ ]:
await star.pipettes.pick_up_tips(
  [full_10.get_item(well) for well in ("A1", "B1", "C1", "D1")]
  + [full_1000.get_item(well) for well in ("E1", "F1", "G1", "H1")]
)

In [ ]:
await star.pipettes.return_tips()

## Four kinds

The return is three commands: the 50 uL and 300 uL tips share a collar height.

In [ ]:
await star.pipettes.pick_up_tips(
  [
    full_10.get_item("A1"),
    full_50.get_item("B1"),
    full_300.get_item("C1"),
    full_1000.get_item("D1"),
  ],
  use_channels=[0, 1, 2, 3],
)

In [ ]:
await star.pipettes.return_tips()

## Channels that are not next to each other, and offsets

In [ ]:
await star.pipettes.pick_up_tips(
  [
    full_10.get_item("A2"),
    full_10.get_item("C2"),
    full_1000.get_item("E2"),
    full_1000.get_item("G2"),
  ],
  use_channels=[0, 2, 4, 6],
)

In [ ]:
await star.pipettes.return_tips()

In [ ]:
from pylabrobot.resources import Coordinate

await star.pipettes.pick_up_tips(
  [
    full_10.get_item("A3"),
    full_10.get_item("B3"),
    full_1000.get_item("C3"),
    full_1000.get_item("D3"),
  ],
  offsets=[
    Coordinate(0.5, 0.5, 0),
    Coordinate(-0.5, 0.5, 0),
    Coordinate(0.5, -0.5, 0),
    Coordinate(0, 0, 0),
  ],
)

In [ ]:
await star.pipettes.return_tips()

## Dropped elsewhere, let go on the deck, discarded

In [ ]:
await star.pipettes.pick_up_tips([full_300.get_item(well) for well in ("A4", "B4", "C4", "D4")])

In [ ]:
# into spots they did not come from
await star.pipettes.drop_tips(
  [spot for spot in holed_300.get_all_items() if not spot.has_tip()][:4]
)

In [ ]:
await star.pipettes.pick_up_tips(
  [spot for spot in holed_1000.get_all_items() if spot.has_tip()][:4]
)

In [ ]:
# let go on the deck, a place per channel
place = full_1000.get_item("A1").get_location_wrt(star.deck, x="c", y="c", z="b")
await star.pipettes.drop_tips(
  [place + Coordinate(0, -30 - 18 * channel, 0) for channel in range(4)]
)

In [ ]:
await star.pipettes.pick_up_tips([full_50.get_item(f"{row}5") for row in "ABCDEFGH"])

In [ ]:
await star.pipettes.discard_tips()

In [ ]:
await star.pipettes.pick_up_tips([full_300.get_item(well) for well in ("A6", "B6", "C6", "D6")])

In [ ]:
await star.pipettes.discard_tips(use_channels=[0, 1])

In [ ]:
await star.pipettes.return_tips(use_channels=[2, 3])

## Where everything ended

In [ ]:
print(await star.pipettes.request_stop_disc_z_positions())
print([star.pipettes.get_mounted_tip(channel) is not None for channel in range(8)])